In [0]:
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)

from delta.tables import DeltaTable


# ============================================================
# 1. CONFIGURATION
# ============================================================

storage_account = "stsupplychain2026xx"

silver_events_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "silver/delivery_events/"
)

silver_shipments_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "silver/shipments/"
)


# ============================================================
# 2. ÉTAT AVANT MERGE
# ============================================================

silver_events_before = (
    spark.read
    .format("delta")
    .load(silver_events_path)
)

count_before = silver_events_before.count()

print(
    "Nombre d'événements AVANT MERGE :",
    count_before
)

print("\n=== E000781 AVANT MERGE ===")

display(
    silver_events_before
    .filter(
        F.col("event_id") == "E000781"
    )
    .select(
        "event_id",
        "shipment_id",
        "event_time",
        "event_status",
        "location_city",
        "source_system",
        "updated_at",
        "batch_id"
    )
)


# ============================================================
# 3. MINI-BATCH DE TEST
# ============================================================

merge_test_events = [

    # --------------------------------------------------------
    # CAS 1 : nouvel événement
    #
    # S0135 existe maintenant dans silver_shipments.
    # => INSERT
    # --------------------------------------------------------
    (
        "E000786",
        "S0135",
        datetime(2026, 1, 12, 12, 0),
        "CREATED",
        "Rabat",
        "Morocco",
        "logistics_app",
        datetime(2026, 1, 12, 12, 0)
    ),

    # --------------------------------------------------------
    # CAS 2 : correction d'un événement existant
    #
    # E000781 existe déjà.
    # On conserve son event_time historique.
    # On corrige ici source_system.
    # updated_at plus récent => UPDATE
    # --------------------------------------------------------
    (
        "E000781",
        "S0131",
        datetime(2026, 1, 10, 10, 0),
        "CREATED",
        "Rabat",
        "Morocco",
        "warehouse_system",
        datetime(2026, 1, 12, 14, 0)
    )
]


# ============================================================
# 4. SCHÉMA
# ============================================================

merge_events_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("shipment_id", StringType(), False),
    StructField("event_time", TimestampType(), False),
    StructField("event_status", StringType(), False),
    StructField("location_city", StringType(), True),
    StructField("location_country", StringType(), True),
    StructField("source_system", StringType(), False),
    StructField("updated_at", TimestampType(), False)
])


# ============================================================
# 5. DATAFRAME SOURCE
# ============================================================

merge_source_base = (
    spark.createDataFrame(
        merge_test_events,
        schema=merge_events_schema
    )

    .withColumn(
        "source_file",
        F.lit("phase4_merge_test")
    )

    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )

    .withColumn(
        "batch_id",
        F.lit("merge_test_003")
    )
)


# ============================================================
# 6. RÉCUPÉRER shipment_time
# ============================================================

shipment_reference = (
    spark.read
    .format("delta")
    .load(silver_shipments_path)
    .select(
        "shipment_id",
        "shipment_time"
    )
)

merge_source = (
    merge_source_base

    .join(
        shipment_reference,
        on="shipment_id",
        how="left"
    )

    # Ordre métier du lifecycle
    .withColumn(
        "event_sequence",
        F.when(
            F.col("event_status") == "CREATED", 1
        )
        .when(
            F.col("event_status") == "PACKED", 2
        )
        .when(
            F.col("event_status") == "SHIPPED", 3
        )
        .when(
            F.col("event_status") == "HUB_SCAN", 4
        )
        .when(
            F.col("event_status") == "OUT_FOR_DELIVERY", 5
        )
        .when(
            F.col("event_status") == "DELIVERED", 6
        )
    )
)

print("\n=== SOURCE DU MERGE ===")

display(
    merge_source
    .orderBy("event_id")
)


# ============================================================
# 7. TABLE DELTA CIBLE
# ============================================================

silver_events_delta = (
    DeltaTable.forPath(
        spark,
        silver_events_path
    )
)


# ============================================================
# 8. MERGE
# ============================================================

(
    silver_events_delta.alias("target")

    .merge(
        merge_source.alias("source"),
        "target.event_id = source.event_id"
    )

    # Correction uniquement si la source est plus récente
    .whenMatchedUpdateAll(
        condition="""
            source.updated_at > target.updated_at
        """
    )

    # Nouvel event
    .whenNotMatchedInsertAll()

    .execute()
)

print("\nMERGE delivery_events terminé.")


# ============================================================
# 9. ÉTAT APRÈS MERGE
# ============================================================

silver_events_after = (
    spark.read
    .format("delta")
    .load(silver_events_path)
)

count_after = silver_events_after.count()

print(
    "Nombre AVANT MERGE :",
    count_before
)

print(
    "Nombre APRÈS MERGE :",
    count_after
)

print(
    "Différence :",
    count_after - count_before
)


# ============================================================
# 10. VÉRIFIER INSERT : E000786
# ============================================================

print("\n=== E000786 : INSERT ===")

display(
    silver_events_after
    .filter(
        F.col("event_id") == "E000786"
    )
    .select(
        "event_id",
        "shipment_id",
        "event_time",
        "event_status",
        "event_sequence",
        "source_system",
        "updated_at",
        "batch_id"
    )
)


# ============================================================
# 11. VÉRIFIER UPDATE : E000781
# ============================================================

print("\n=== E000781 : UPDATE ===")

display(
    silver_events_after
    .filter(
        F.col("event_id") == "E000781"
    )
    .select(
        "event_id",
        "shipment_id",
        "event_time",
        "event_status",
        "event_sequence",
        "source_system",
        "updated_at",
        "batch_id"
    )
)